# Informe de Selección de Modelos — [nombre del conjunto de datos]

- **Grupo**: Carolina Giraldo Toro CC
Yirlen Dayanny Mosquera Lozano CC1193489556
- **Variante**: clasificación binaria
- **Conjunto de datos**: [nombre, con un enlace a su fuente]
- **Fecha**: [2026-10-03]


🗑️ **Elimine esta sección antes de entregar** — contiene instrucciones, no forma parte de su informe.

## Comience Aquí

Este cuaderno es un **ejemplo práctico completo** de un flujo de trabajo: seleccionar, ajustar y probar un
modelo de clasificación. Se ejecuta sobre un conjunto de datos de cancelación de clientes (churn). Su grupo repetirá el mismo flujo de trabajo en un
conjunto de datos que ustedes mismos elijan. **Copie este archivo, conserve su estructura y reemplace el contenido de cancelación
con el suyo.** La copia es lo que debe entregar.

**Lo que practicará:**

1. Describir un conjunto de datos y verificar que se ajuste a la tarea.
2. Elegir una métrica antes de ver cualquier resultado y usarla en todas partes.
3. Dividir los datos en un conjunto de entrenamiento y un conjunto de prueba, y definir los pliegues de validación cruzada una sola vez.
4. Construir un pipeline de preprocesamiento que aprenda únicamente de las filas con las que se ajusta.
5. Comparar una línea base y cuatro familias de modelos mediante validación cruzada, y elegir uno según una regla.
6. Ajustar el modelo elegido con los mismos pliegues y la misma métrica.
7. Medir el modelo final una sola vez en el conjunto de prueba y comparar ese número con la estimación.

**El flujo de trabajo, en una imagen:**

```
datos ─► división ─┬─► conjunto de entrenamiento ─► CV de 5 pliegues ─► torneo ─► modelo elegido ─► ajuste ─┐
                   │                                (línea base + 4 familias)     (mismo CV)                │
                   └─► conjunto de prueba ── apartado, no se usa ────────────────────────────► una prueba final
```

**Su informe consta de ocho ✍️ celdas**, una por cada paso:

| # | Entregable | Sección |
|---|---|---|
| 1 | Ficha del conjunto de datos | 1 |
| 2 | La métrica | 2 |
| 3 | La división | 3 |
| 4 | La tabla de preprocesamiento | 4 |
| 5 | El torneo | 5 |
| 6 | El ajuste | 6 |
| 7 | La prueba final | 7 |
| 8 | Conclusiones | 8 |

**Qué es cada tipo de celda y qué debe hacer con ella:**

| Celda | Qué es | Qué debe hacer |
|---|---|---|
| ✍️ **Entregable** | una pregunta para su grupo | respóngala sobre **sus** datos; consérvela |
| 💡 **Ejemplo** | nuestra respuesta para los datos de cancelación | léala y luego elimínela |
| 🗑️ **Guía** | qué hacer en una sección y 🎯 qué verificamos | léala y luego elimínela |
| celdas de código | el flujo de trabajo | consérvelas; cámbielas para sus datos |
| otro markdown | la oración antes de cada celda de código y la lectura posterior | consérvelas; reescríbalas sobre sus datos y sus números |

**Reglas aplicables en todas las secciones. Cada una se verifica al calificar:**

1. **El conjunto de prueba se usa una sola vez**, en la Sección 7, y nunca antes.
2. **Una métrica** se usa para cada comparación: torneo, ajuste y prueba.
3. **Una semilla aleatoria**, definida en la Sección 0, se pasa a cada paso que utilice aleatoriedad.
4. **Cada número en su texto proviene de una salida en su cuaderno.** No copie nuestros números.
5. **Cada celda de código tiene una oración antes de ella** indicando lo que hace la celda.
6. **Ejecute todo el cuaderno de arriba a abajo** (*Reiniciar Kernel y Ejecutar Todas las Celdas*) antes de
   entregar, y entréguelo **con sus salidas**.

**Su conjunto de datos debe cumplir con estos requisitos mínimos. La Sección 1 los verifica mediante código:**

- un **objetivo binario** (dos clases), donde la clase minoritaria represente el **20% de las filas o más**;
- al menos **1,000 filas**;
- al menos **8 columnas predictoras**, incluyendo tanto columnas numéricas como categóricas;
- al menos una necesidad real de preprocesamiento más allá del escalado, como valores faltantes o columnas categóricas
  por codificar;
- **no** debe ser uno de los conjuntos de datos del curso (cancelación de clientes, viviendas de Ames, iris, seguros, MPG de automóviles).

Dos grupos pueden usar el mismo conjunto de datos, bajo su propio riesgo.

**Una limitación conocida de antemano.** El torneo en la Sección 5 compara modelos con **configuraciones
iniciales** fijadas por reglas simples, no ajustadas, y solo el ganador es ajustado. Un modelo aún puede
perder una comparación que podría ganar después de ser ajustado. Esto se acepta deliberadamente para mantener el flujo de trabajo breve.
Menciónelo en sus conclusiones; no necesita solucionarlo.

**Cómo se califica su trabajo.** Cada ✍️ celda se evalúa sobre su propio conjunto de datos. **No** comparamos puntajes
entre grupos. Un puntaje modesto en un problema difícil, reportado honestamente, vale más que un puntaje alto
derivado de un error. La guía 🗑️ de cada sección termina con una lista 🎯: esa lista es lo que buscamos
en dicha sección.

## 0. Configuración

La celda de abajo importa todas las bibliotecas que utiliza el cuaderno y fija la semilla aleatoria. Si se ejecuta
sin errores, el entorno está listo:

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

## 1. Los Datos

### 1.1 Carga del archivo bruto

La celda de abajo lee el archivo bruto e imprime su tamaño:

In [7]:
DATA_PATH = "dataset_limpio.csv"

df = pd.read_csv(DATA_PATH)
print(f"{df.shape[0]:,} rows x {df.shape[1]} columns")

448,828 rows x 22 columns


Los datos tienen 448.828 filas y 22 columnas. **Una fila es un cliente por mes**, la data es de tres meses, de una empresa de telecomunicaciones.
Para efectos del proceso a realizar nos enfocaremos unicamente en el ultimo mes y de la cantidad de registros resultantes, tomaremos una muestra de 20.000 registros, los cuales, serán seleccionados especificamente para conservar la proporción de clientes con churned y sin el, porque esta es nuestra columna objetivo.


In [8]:
# 1. Filtrar el DataFrame original para conservar únicamente el último mes disponible
ultimo_mes = df['mes'].max()
df_ultimo_mes = df[df['mes'] == ultimo_mes].copy()

# 2. Definir parámetros de la muestra
n_total = 20000
prop_minority = 0.20  # 20% para la clase minoritaria
n_minority = int(n_total * prop_minority)  # 4,000 registros para churned = 1
n_majority = n_total - n_minority          # 16,000 registros para churned = 0

# 3. Separar el subconjunto del último mes en clase mayoritaria (0) y minoritaria (1)
df_majority = df_ultimo_mes[df_ultimo_mes['churned'] == 0]
df_minority = df_ultimo_mes[df_ultimo_mes['churned'] == 1]

# 4. Extraer las muestras aleatorias usando RANDOM_STATE
df_majority_sample = df_majority.sample(n=n_majority, random_state=RANDOM_STATE)
df_minority_sample = df_minority.sample(n=n_minority, random_state=RANDOM_STATE)

# 5. Unir ambas muestras, reasignarlas a 'df' y mezclarlas aleatoriamente
df = pd.concat([df_majority_sample, df_minority_sample]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# 6. Verificar el tamaño, clientes únicos y distribución de clases
print(f"Tamaño final de la muestra: {len(df)}")
print(f"Clientes únicos (user_id distintos): {df['user_id'].nunique()}")
print("\nConteo por clase:")
print(df['churned'].value_counts())
print("\nProporción por clase:")
print(df['churned'].value_counts(normalize=True))

Tamaño final de la muestra: 20000
Clientes únicos (user_id distintos): 20000

Conteo por clase:
churned
0    16000
1     4000
Name: count, dtype: int64

Proporción por clase:
churned
0    0.8
1    0.2
Name: proportion, dtype: float64


Proporción de la clase minoritaria: La clase 1 (churned) tiene 4,000 registros, representando exactamente el 20% de la muestra total, cumpliendo con la regla de que la clase minoritaria sea del 20% o más.

La siguiente celda muestra las primeras filas para que pueda ver cómo es un cliente en los datos:

In [9]:
df.head()

,user_id,mes,pais,ciudad,localidad,latitud,longitud,signup_date,tenure_months,es_cliente_nuevo_mes,...,complaints_month,nps_score,churned,churn_type,churn_date,region_n,tier_n,plan_type_n,es_outlier,revenue_segment
0,CNX-192531,2026-05-01,Colombia,Barranquilla,La Floresta,10.975034,-74.770658,2024-08-13,21,False,...,1,NaN,0,NaN,NaN,Caribe,Media,Pospago,False,Medio
1,CNX-190411,2026-05-01,Colombia,Barranquilla,Ciudad Jardín,10.923752,-74.848151,2024-05-19,24,False,...,0,NaN,0,NaN,NaN,Caribe,Alta,Pospago,False,Alto
2,CNX-254948,2026-05-01,Colombia,Arauca,Villa del Río,7.101205,-70.721006,2026-04-22,1,False,...,2,NaN,0,NaN,NaN,Orinoquía,Media,Convergente,False,Alto
3,CNX-114444,2026-05-01,Colombia,Medellín,Cabecera,6.230299,-75.555339,2020-01-15,76,False,...,1,NaN,0,NaN,NaN,Andina,Alta,Pospago,False,Premium
4,CNX-119068,2026-05-01,Colombia,Medellín,San Fernando,6.268406,-75.624242,2025-06-28,11,False,...,0,18.1,0,NaN,NaN,Andina,Media,Convergente,False,Premium


In [10]:
pd.DataFrame(
    {
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "distinct values": df.nunique(),
    }
)

,type,missing,distinct values
user_id,str,0,20000
mes,str,0,1
pais,str,0,1
ciudad,str,0,25
localidad,str,0,39
latitud,float64,399,19167
longitud,float64,399,18996
signup_date,str,0,2190
tenure_months,int64,0,101
es_cliente_nuevo_mes,bool,0,2


Es importante resaltar que `user_id` tiene 20.000 valores distintos, uno por fila: es un
**identificador**, no información sobre el cliente. `churned` solo tiene dos valores distitos porque los únicos valores posibles son 0 para No churn y 1 para churn. Las columnas `mes` y `signup_date` se corregirán a tipo date.

### 1.2 Correcciones estructurales

A continuación abordaremos las siguientas correcciones estructurales:

- Eliminación de Fugas: columnas churn_type y churn_date
- Limpieza de Identificadores: columna user_id.
- Tipos de datos: correcta lectura de fechas y eliminación variables constantes (mes)

In [15]:
# Crear una copia para las correcciones estructurales
df_estructurado = df.copy()

# 1. Eliminar identificadores y columnas con fugas de datos
# - 'user_id': Es un identificador único; no aporta un patrón predictivo general y puede causar sobreajuste.
# - 'churn_type' y 'churn_date': Contienen información sobre el motivo y la fecha exacta del churn, lo cual ocurre después del hecho y genera una grave fuga de datos.
# - 'mes': Como filtramos previamente para quedarnos con un único mes, ya no aporta variabilidad al modelo.
# - 'es_outlier': Esta columna indica si un registro es un outlier, lo cual es información que no estaría disponible en un escenario de predicción real y podría inducir a errores en el modelo.    
columnas_a_eliminar = ['user_id', 'churn_type', 'churn_date', 'mes', 'es_outlier']

df_estructurado = df_estructurado.drop(columns=[col for col in columnas_a_eliminar if col in df_estructurado.columns])

# 2. Corrección de tipos de columnas (ej. convertir fechas si es necesario para el análisis previo)
if 'signup_date' in df_estructurado.columns:
    df_estructurado['signup_date'] = pd.to_datetime(df_estructurado['signup_date'], errors='coerce')

# 3. Verificar las columnas y tipos resultantes
print("Dimensiones finales tras correcciones estructurales:", df_estructurado.shape)
display(df_estructurado.dtypes)

Dimensiones finales tras correcciones estructurales: (20000, 17)


pais                               str
ciudad                             str
localidad                          str
latitud                        float64
longitud                       float64
signup_date             datetime64[us]
tenure_months                    int64
es_cliente_nuevo_mes              bool
monthly_revenue_cop            float64
data_usage_gb                  float64
complaints_month                 int64
nps_score                      float64
churned                          int64
region_n                           str
tier_n                             str
plan_type_n                        str
revenue_segment                    str
dtype: object

- Eliminación de Fugas: Se descartaron `churn_type` y `churn_date` para evitar que el modelo "haga trampa" viendo datos del futuro. 

- Eliminación de información no disponible: Se eliminó `es_outlier` lo cual es información que no estaría disponible en un escenario de predicción real y podría inducir a errores en el modelo.    

- Limpieza de Identificadores: Se removió `user_id` para que el algoritmo aprenda comportamientos de los clientes y no memorice identificaciones específicas.

- Tipos de datos: Se aseguró la correcta lectura de fechas y se eliminaron variables constantes (mes), dejando el DataFrame listo para la división entre entrenamiento y prueba.

In [16]:
# Definir la variable objetivo
TARGET = "churned"

# Verificar que la variable objetivo esté lista
print(f"Variable objetivo definida: {TARGET}")
print("Distribución de clases en el objetivo:")
print(df_estructurado[TARGET].value_counts())

Variable objetivo definida: churned
Distribución de clases en el objetivo:
churned
0    16000
1     4000
Name: count, dtype: int64


In [17]:
# Contar los registros por cada clase en la variable objetivo
counts = df_estructurado[TARGET].value_counts()

# Crear y mostrar la tabla resumen con el conteo absoluto y la proporción
pd.DataFrame({
    "customers": counts, 
    "share": (counts / len(df_estructurado)).round(4)
})

,customers,share
churned,,
0,16000,0.8
1,4000,0.2


El 80% de los clientes no hicieron churn, es decir, continuaron con el servicio: esa es la clase 0, la clase positiva. El 20% restante se retiro.

### 1.4 Verificación de requisitos mínimos

La celda de abajo verifica cada requisito mínimo de *Comience Aquí* e imprime `PASS` o `FAIL`. Consérvela
en su informe: es la evidencia para el Entregable 1.

In [18]:
# Definir los predictores excluyendo la variable objetivo
predictors = df_estructurado.drop(columns=TARGET)

# Calcular métricas para las validaciones
n_numeric = predictors.select_dtypes(include="number").shape[1]
n_categorical = predictors.shape[1] - n_numeric
minority_share = df_estructurado[TARGET].value_counts(normalize=True).min()
n_missing = int(predictors.isna().sum().sum())

# Diccionario con los requisitos mínimos del taller
requirements = {
    "binary target": df_estructurado[TARGET].nunique() == 2,
    f"smaller class >= 20% (it is {minority_share:.1%})": minority_share >= 0.20,
    f"at least 1,000 rows (it has {len(df_estructurado):,})": len(df_estructurado) >= 1000,
    f"at least 8 predictors (it has {predictors.shape[1]})": predictors.shape[1] >= 8,
    f"numeric and categorical columns ({n_numeric} and {n_categorical})": n_numeric > 0 and n_categorical > 0,
    f"a preprocessing need ({n_missing} missing, {n_categorical} to encode)": n_missing > 0 or n_categorical > 0,
}

# Evaluar e imprimir el resultado de cada requisito
for requirement, passed in requirements.items():
    print(f"{'PASS' if passed else 'FAIL'} {requirement}")

PASS binary target
PASS smaller class >= 20% (it is 20.0%)
PASS at least 1,000 rows (it has 20,000)
PASS at least 8 predictors (it has 16)
PASS numeric and categorical columns (7 and 9)
PASS a preprocessing need (15821 missing, 9 to encode)


Todas las líneas pasan, es decir, se cumple con los requisitos planteados el el inicio.

✍️ **Entregable 1 de 8 — Ficha del conjunto de datos**

- **Nombre y fuente**: *Conecta S.A Gruo 8*, es un conjunto de datos de ejemplo académico de una empresa de telecomunicaciones, el cual pasó por un proceso de limpieza de datos y EDA, enlace.
- **Qué representa una fila**: un cliente de una empresa de telecomunicaciones: donde estan ubicados, qué
  servicios tienen contratados y cómo se factura su cuenta.
- **Tamaño después de las correcciones estructurales**:  20.000 filas × 17 columnas (16 predictoras y el objetivo).
- **El objetivo**: `churned`. 1 significa que el cliente se fue en el último mes, 0 significa que se quedó. El 80% de
  los clientes son positivos.
- **La decisión**: a qué clientes debe contactar primero el equipo de retención con una oferta. El modelo
  ordena a los clientes según su riesgo de irse.
- **Columnas eliminadas**: `user_id`, un identificador con un valor por fila; un modelo solo podría
  memorizarlo. Verificamos si había columnas con fugas: `churn_type`y `churn_date`.
  `mes` no aportaba información relevante porque todas corresponden al mes de mayo, así como `es_outlier` en un escenario real no podría estar y generar error al modelo.
- **Conversión de tipo de datos**: la columna `signup_date` pasó de formato str a dateTime como correspondía
    
    Ningún paso calcula nada a partir de otras filas.
- **Verificación de requisitos**: cada línea imprime `PASS`.

## 2. La Métrica

La métrica seleccionada para todo el proyecto es ROC AUC (`SCORING = "roc_auc"`).

La decisión de negocio consiste en priorizar qué clientes deben ser contactados por el equipo de retención. En esta etapa aún no se dispone de información sobre costos de contacto, costos de incentivos o valor económico de los clientes retenidos, por lo que no es posible definir un umbral de clasificación adecuado. Debido a ello, se requiere una métrica que permita evaluar la capacidad del modelo para distinguir entre clientes que abandonan y clientes que permanecen sin depender de un umbral específico.

ROC AUC resulta apropiada porque evalúa el ordenamiento de los clientes según su riesgo de churn considerando todos los posibles umbrales de decisión. Esto permite comparar modelos antes de definir la estrategia de negocio que determinará el umbral final.
 
> **En `scikit-learn`.** `scoring="roc_auc"`; las puntuaciones provienen de `predict_proba` o `decision_function`.

Para evaluar todos los modelos se utilizará la métrica ROC AUC, definida en el código mediante la constante:

In [19]:
SCORING = "roc_auc"

✍️ **Entregable 2 de 8 — La métrica**

- **La métrica**: ROC AUC. Se utilizará en todas las etapas del proyecto (selección de modelos, validación y evaluación final), manteniendo la misma métrica mediante la constante SCORING = "roc_auc".
- **Qué significarían una puntuación de 0.5 y una de 1.0 para su problema:** Un ROC AUC de 0.5 significa que el modelo no distingue mejor que el azar entre los clientes que abandonarán el servicio (churned = 1) y los que permanecerán (churned = 0). En otras palabras, al ordenar a los clientes por riesgo de churn, el modelo no ubica a los clientes que se irán por encima de los que se quedarán con mayor frecuencia que una selección aleatoria. Mientras que un ROC AUC de 1.0 significa que el modelo ordena perfectamente a los clientes según su riesgo de abandono. Todos los clientes que efectivamente hicieron churn reciben una puntuación de riesgo superior a la de cualquier cliente que no hizo churn.
- **Qué significaría un umbral en su problema**: El umbral sería el valor a partir del cual un cliente se considera de alto riesgo de churn. Los clientes cuya puntuación supere ese umbral serían priorizados por el equipo de retención para ser contactados y recibir ofertas o acciones de fidelización. Los clientes que queden por debajo del umbral no serían contactados en esa campaña.
- **De qué dependería el umbral** y por qué se deja para más adelante: El umbral dependería de factores de negocio que aún no conocemos, como el costo de contactar a un cliente, el costo de las ofertas de retención, el valor económico de conservar un cliente y la capacidad operativa del equipo de retención. Debido a que estas variables no forman parte del problema de modelado y serán definidas por el negocio, en esta etapa nos enfocamos en construir un modelo que ordene correctamente a los clientes según su riesgo de churn. Por ello utilizamos ROC AUC, ya que evalúa la calidad de esa clasificación sin necesidad de fijar previamente un umbral de decisión.

## 3. División de los Datos

🗑️ **Guía — Sección 3.** Elimine antes de entregar.

Esta sección realiza ambas divisiones en un solo lugar:

- **El conjunto de prueba apartado (hold-out)**: 20% de las filas, estratificado según el objetivo y con semilla. No se vuelve a usar
  hasta la Sección 7: cada revisión previa de este conjunto hace que su puntuación final sea menos honesta.
- **Los pliegues de validación cruzada**: un objeto `StratifiedKFold`, mezclado (shuffled) y con semilla, pasado como `cv=`
  a cada paso de validación cruzada en las Secciones 5 y 6, para que cada modelo sea evaluado con las mismas filas.

🔧 **Para sus datos:** por lo general, nada. Conserve el 20% para el conjunto de prueba y 5 pliegues, estratificados.

🎯 **Lo que verificamos:**
- La división ocurre antes de cualquier paso que aprenda de los datos.
- Ambas divisiones están estratificadas y tienen semilla.
- Un solo objeto de pliegues, creado aquí y pasado a **cada** paso de validación cruzada.
- La proporción de clases impresa para los conjuntos de entrenamiento y prueba.
- El conjunto de prueba no se vuelve a utilizar hasta la Sección 7.

### 3.1 El conjunto de prueba apartado

La celda de abajo separa los predictores del objetivo y aparta el 20% de las filas como conjunto de
prueba. Estratificar según el objetivo mantiene la proporción de cancelaciones igual en ambas partes:

In [20]:
X = df_estructurado.drop(columns=TARGET)
y = df_estructurado[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)

La siguiente celda verifica el tamaño de cada parte y su proporción de filas positivas:

In [21]:
pd.DataFrame(
    {
        "rows": [len(y_train), len(y_test)],
        "positive share": [y_train.mean(), y_test.mean()],
    },
    index=["training set", "test set"],
).round(4)

,rows,positive share
training set,16000,0.2
test set,4000,0.2


El conjunto de entrenamiento tiene 16.000 filas y el conjunto de prueba 4.000, y ambos conservan la proporción del 20% de clientes que cancelaron.
**Desde aquí hasta la Sección 7, `X_test` y `y_test` no se utilizan.**

### 3.2 Los pliegues de validación cruzada

> **En `scikit-learn`.** `StratifiedKFold` define pliegues estratificados. `shuffle=True` mezcla las filas
> antes de dividirlas, para que cualquier orden en el archivo (por ejemplo, por fecha) no determine los
> pliegues; la mezcla utiliza aleatoriedad, por lo que requiere la semilla. El objeto se pasa luego como `cv=` a cada
> función de validación cruzada.

La celda de abajo crea el objeto de pliegues. Se crea aquí una sola vez y se pasa a cada paso de validación
cruzada en las Secciones 5 y 6:

In [22]:
N_FOLDS = 5
cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

La siguiente celda enumera los cinco pliegues, con el número de filas con las que se ajusta cada modelo, el tamaño del
pliegue de validación y su proporción de filas positivas:

In [23]:
fold_rows = []
for fold, (fit_index, validation_index) in enumerate(cv.split(X_train, y_train), start=1):
    fold_rows.append(
        {
            "fold": fold,
            "rows fitted on": len(fit_index),
            "validation rows": len(validation_index),
            "validation positive share": y_train.iloc[validation_index].mean(),
        }
    )

pd.DataFrame(fold_rows).set_index("fold").round(4)

,rows fitted on,validation rows,validation positive share
fold,,,
1,12800,3200,0.2
2,12800,3200,0.2
3,12800,3200,0.2
4,12800,3200,0.2
5,12800,3200,0.2


Cada modelo se ajusta con aproximadamente 12.800 clientes y se evalúa con 3.200 clientes en cada pliegue de validación cruzada. En todos los pliegues se conserva una proporción del 20% de clientes que realizaron churn y un 80% de clientes que permanecieron, asegurando que los conjuntos de entrenamiento y validación sean representativos de la distribución original de los datos. Los mismos cinco pliegues se utilizan para todos los modelos, permitiendo una comparación justa de su capacidad para identificar clientes con riesgo de abandono.

✍️ **Entregable 3 de 8 — La división**

- **Tamaños**: El conjunto de datos utilizado para el modelado contiene 20.000 registros. De ellos, el 80% (16.000 filas) se destinó al conjunto de entrenamiento y el 20% (4.000 filas) al conjunto de prueba.
- **Por qué 20%**: Se seleccionó 4.000 filas porque proporciona una cantidad suficiente de datos para entrenar los modelos y, al mismo tiempo deja un conjunto independiente de observaciones para evaluar su desempeño final. De esta forma, el modelo puede aprender con una muestra amplia y posteriormente ser evaluado sobre datos que no utilizó durante el entrenamiento. 
- **Pliegues**: Se utilizó validación cruzada estratificada de 5 pliegues. En cada iteración, el modelo se ajusta con aproximadamente 12.800 filas y se evalúa con 3.200 filas. Los mismos pliegues se emplean para todos los modelos, permitiendo una comparación consistente entre alternativas.
- **Por qué estratificada**: los que churnean son el 20% de las filas. La estratificación mantiene esa proporción en el conjunto de prueba y en cada pliegue, de modo que ninguna puntuación varíe porque una parte recibió más o menos churneadores por casualidad.
- **Regla del conjunto de prueba**: `X_test` y `y_test` a partir de este punto, el conjunto de prueba permanecerá completamente separado del proceso de entrenamiento y selección de modelos. Se utilizará únicamente para la evaluación final del modelo elegido. Todas las decisiones de modelado, ajuste de hiperparámetros y comparación entre algoritmos se realizarán exclusivamente utilizando el conjunto de entrenamiento y la validación cruzada.

## 4. El Pipeline de Preprocesamiento

🗑️ **Guía — Sección 4.** Elimine antes de entregar.

- Coloque cada paso que calcule un valor a partir de los datos (imputadores, escaladores, codificadores) en el
  preprocesador y únalo a cada modelo con `make_model`. La validación cruzada lo reajustará en los
  pliegues de ajuste de cada ronda.
- Nunca pase una matriz preprocesada a la validación cruzada: pase siempre el `X_train` sin procesar y el
  pipeline completo. Preprocesar todo el conjunto de entrenamiento primero permite que cada pliegue de validación modele las medianas
  y medias con las que se entrena el modelo. Eso es fuga de datos (leakage); su efecto suele ser pequeño, por lo que pasa
  desapercibido.
- Utilice un solo preprocesador para cada candidato.
- Escale las columnas numéricas: KNN y la regresión logística lo necesitan.

🔧 **Para sus datos:** los grupos de columnas y el tratamiento de cada grupo. Tratamientos adicionales comunes:
- una transformación logarítmica para una columna numérica fuertemente sesgada;
- una codificación ordinal para categorías que tienen un orden natural;
- agrupar categorías raras en una columna con muchísimas categorías.

Cada tratamiento que agregue debe estar justificado en el Entregable 4.

🎯 **Lo que verificamos:**
- Cada columna predictora pertenece exactamente a un grupo (la celda de verificación se ejecuta correctamente).
- El preprocesamiento ocurre únicamente dentro del pipeline: no hay `fit_transform` cuyo resultado se envíe a la validación cruzada.
- Nada se ajusta con el conjunto de prueba.
- El Entregable 4 cubre cada grupo y justifica cada tratamiento con sus datos.

### 4.1 Grupos de columnas

La celda de abajo asigna cada columna predictora a un grupo por su nombre:

In [ ]:
NUMERIC_COLS = ["tenure", "MonthlyCharges", "TotalCharges"]
BINARY_COLS = ["SeniorCitizen"]
CATEGORICAL_COLS = [
    column for column in X_train.columns if column not in NUMERIC_COLS + BINARY_COLS
]

print(f"numeric ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"binary ({len(BINARY_COLS)}): {BINARY_COLS}")
print(f"categorical ({len(CATEGORICAL_COLS)}): {CATEGORICAL_COLS}")

Una columna que quede fuera de todos los grupos sería **eliminada sin previo aviso**. La celda siguiente verifica que
cada predictor esté exactamente en un grupo y detiene el cuaderno si alguno no lo está:

In [ ]:
assigned = NUMERIC_COLS + BINARY_COLS + CATEGORICAL_COLS

assert len(assigned) == len(set(assigned)), "a column is in more than one group"
assert set(assigned) == set(X_train.columns), "a column is in no group, or a name is misspelled"
print(f"all {len(assigned)} predictor columns are in exactly one group")

### 4.2 Tratamiento de cada grupo

Los tratamientos convencionales son los siguientes:

- **Columnas numéricas.** Los valores faltantes se rellenan con la **mediana** de la columna en las filas de ajuste.
  Luego, cada columna es **estandarizada**.
  Las columnas estandarizadas tienen media 0 y se miden en desviaciones estándar. Estas columnas no
  tienen un rango fijo, pero la mayoría de los valores se sitúan entre −3 y 3.

- **Columnas binarias** ya son 0/1 y se mantienen tal cual.
- **Columnas categóricas.** Los valores faltantes se rellenan con la categoría más frecuente. Luego, la **codificación
  one-hot** convierte cada columna en una columna 0/1 por categoría: `Contract` se convierte en tres columnas,
  una para cada tipo de contrato.

> **En `scikit-learn`.** `ColumnTransformer` envía cada grupo a sus propios pasos y, por defecto,
> **elimina** cualquier columna que no figure en un grupo (`remainder="drop"`); por eso 4.1 verifica los grupos.
> `SimpleImputer`, `StandardScaler` y `OneHotEncoder` realizan los pasos anteriores.
> `handle_unknown="ignore"` hace que el codificador escriba todo ceros para una categoría que no vio durante
> el ajuste, en lugar de detenerse con un error; una categoría poco común puede estar ausente en los pliegues de ajuste de
> una ronda. `set_output(transform="pandas")` conserva los nombres de las columnas en la salida.

La celda de abajo construye el preprocesador a partir de los tres grupos. Aún no se ajusta nada:

In [ ]:
numeric_steps = Pipeline(
    [
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]
)
categorical_steps = Pipeline(
    [
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_steps, NUMERIC_COLS),
        ("binary", "passthrough", BINARY_COLS),
        ("categorical", categorical_steps, CATEGORICAL_COLS),
    ],
    verbose_feature_names_out=False,
).set_output(transform="pandas")

La siguiente celda define `make_model`, que une el preprocesador y un modelo en un solo pipeline.
Cada modelo en las Secciones 5, 6 y 7 se construye con él, de modo que cada modelo recibe exactamente el mismo
preprocesamiento:

In [ ]:
def make_model(estimator: BaseEstimator) -> Pipeline:
    """Une el preprocesador compartido y un modelo en un solo pipeline.

    Args:
        estimator: Cualquier clasificador de scikit-learn, aún no ajustado.

    Returns:
        Un pipeline que prepara las columnas y luego aplica el modelo.
    """
    return Pipeline([("prep", preprocessor), ("model", estimator)])

La siguiente celda ajusta el preprocesador en el conjunto de entrenamiento **solo para observar su salida**: cuántas
columnas recibirán los modelos y cómo se ven:

In [ ]:
X_train_preproc = preprocessor.fit_transform(X_train)

print(f"{X_train.shape[1]} columns in, {X_train_preproc.shape[1]} columns out")
X_train_preproc.head(3)

Los 19 predictores se convierten en 45 columnas, la mayoría de ellas columnas one-hot provenientes de los 15 predictores categóricos.
Las columnas numéricas ahora tienen valores cercanos a 0.

> **`X_train_preproc` existe solo para ser visualizado.** Fue preparado con estadísticas de todo el
> conjunto de entrenamiento, por lo que nunca debe pasarse a la validación cruzada. A partir de la Sección 5, la validación cruzada
> recibe siempre el `X_train` sin procesar y el pipeline completo.

✍️ **Entregable 4 de 8 — La tabla de preprocesamiento**

Una fila por grupo de columnas:

| Grupo | Columnas | Tratamiento, en orden | Por qué, para sus datos |
|---|---|---|---|
| | | | |

Luego responda:
- **Dónde se encuentran sus valores faltantes** y cómo los rellena el pipeline: *...*
- **El número de columnas** antes y después del preprocesamiento: *...*
- **Por qué el preprocesamiento está dentro del pipeline**, en una o dos oraciones: *...*

💡 **Ejemplo** — Entregable 4 para los datos de cancelación. Elimínelo antes de entregar.

| Grupo | Columnas | Tratamiento, en orden | Por qué, para sus datos |
|---|---|---|---|
| Numéricas | `tenure`, `MonthlyCharges`, `TotalCharges` | rellenar con mediana y luego estandarizar | `TotalCharges` tiene 11 valores faltantes, y la mediana no se ve afectada por la larga cola derecha de los cargos; el escalado coloca los meses y el dinero en una sola escala para KNN y regresión logística |
| Binarias | `SeniorCitizen` | ninguno | ya son 0/1 |
| Categóricas | las otras 15 columnas | rellenar con el más frecuente y luego one-hot | no hay valores faltantes hoy, pero el relleno no cuesta nada y protege contra un espacio en blanco en datos nuevos; one-hot porque las categorías (tipo de contrato, método de pago) no tienen un orden |

- **Valores faltantes**: solo los 11 clientes nuevos en `TotalCharges`. El pipeline los rellena con la
  mediana de las filas de ajuste. Rellenar con 0 también sería defendible, ya que aún no se les ha
  facturado. Con 11 filas de 7,043, la elección no puede alterar los resultados.
- **Columnas**: 19 predictores de entrada, 45 columnas de salida.
- **Por qué dentro del pipeline**: la validación cruzada reajusta el pipeline en cada ronda, de modo que las medianas,
  medias y listas de categorías provienen únicamente de los pliegues de ajuste, y el pliegue de validación permanece sin ver.

## 5. El Torneo de Modelos

🗑️ **Guía — Sección 5.** Elimine antes de entregar.

Evalúe a cada candidato con los mismos pliegues, la misma métrica y el mismo preprocesamiento.

- Incluya la línea base y un modelo de cada una de las cuatro familias.
- Inicie cada familia a partir de la regla de la tabla siguiente. La celda en 5.1 calcula los valores a partir del tamaño
  de sus datos. Fíjelos antes de que se ejecute el torneo y no los modifique después de ver una puntuación: eso es ajustar
  manualmente, lo cual nadie registra y hace que la comparación con la Sección 6 sea injusta.
- El modelo con el AUC medio de validación cruzada más alto pasa a la Sección 6.

**Las cuatro familias requeridas**, con sus reglas iniciales y los hiperparámetros que más importan cuando
se ajusta esa familia. $n$ es el número de filas con las que se ajusta cada modelo en un pliegue:

| Familia | Nuestro modelo | Configuración inicial | Principales hiperparámetros a ajustar |
|---|---|---|---|
| Lineal regularizada | regresión logística | valor predeterminado de la biblioteca ($C = 1$) | intensidad de regulación; mezcla de penalizaciones L1 y L2 |
| Vecinos más cercanos | KNN | $k = \sqrt{n}$, redondeado a un número impar | número de vecinos; si los vecinos más cercanos cuentan más |
| Árbol único | árbol de decisión | profundidad máxima 5; al menos 1% de $n$ filas por hoja | profundidad máxima; número mínimo de filas por hoja |
| Ensamblaje | gradient boosting | valor predeterminado de la biblioteca (100 árboles, profundidad 3, tasa de aprendizaje 0.1) | número de árboles; tasa de aprendizaje; profundidad del árbol |

Si su ensamblaje es un **bosque aleatorio (random forest)**, inícielo con al menos 5 filas por hoja.

🔧 **Para sus datos:** nada en las reglas; la celda de configuraciones iniciales calcula los valores a partir
de su recuento de filas. Puede agregar candidatos, por ejemplo un segundo ensamblaje, cada uno con una regla
inicial declarada. No puede eliminar la línea base ni ninguna de las cuatro familias.

🎯 **Lo que verificamos:**
- La línea base y las cuatro familias están presentes.
- Cada configuración inicial proviene de una regla declarada, calculada en código, sin indicios de cambios realizados
  después de ver las puntuaciones.
- El mismo objeto de pliegues y la misma métrica en cada llamada.
- Una tabla de resultados con la media y la dispersión de las puntuaciones de validación.
- La regla de selección declarada antes de los resultados.

### 5.1 Los candidatos y sus configuraciones iniciales

Las reglas iniciales para KNN y el árbol de decisión necesitan $n$, el número de filas con las que se ajusta cada modelo
en un pliegue. La celda de abajo calcula $n$ a partir del conjunto de entrenamiento y el objeto de pliegues, aplica las dos
reglas e imprime los valores que utilizarán los candidatos:

In [ ]:
fit_rows = len(X_train) * (N_FOLDS - 1) // N_FOLDS

k_neighbors = round(np.sqrt(fit_rows))

# If k_neighbors is even -> k_neighbors + 1
if k_neighbors % 2 == 0:
    k_neighbors += 1
    
min_leaf = round(0.01 * fit_rows)

print(f"rows fitted on in one fold (n): {fit_rows:,}")
print(f"KNN: k = sqrt(n), rounded to an odd number -> {k_neighbors}")
print(f"decision tree: maximum depth 5, at least 1% of n per leaf -> {min_leaf} rows")

Cada modelo se ajusta con 4,507 filas, por lo que KNN comienza con $k = 67$ vecinos y el árbol con al menos
45 filas en cada hoja. En un conjunto de datos de un tamaño diferente, las mismas reglas
proporcionan valores diferentes.

> **En `scikit-learn`.** `LogisticRegression` está regularizado por defecto: una penalización L2 con una intensidad
> establecida por `C=1.0`, donde un `C` **más pequeño** significa una penalización **más fuerte**. `max_iter=1000` otorga al
> optimizador suficientes iteraciones para terminar. `DummyClassifier(strategy="most_frequent")` es la línea base:
> otorga a cada fila la misma puntuación, por lo que su AUC es 0.5 por construcción. Para KNN, el número de vecinos
> es `n_neighbors`; para el árbol, `max_depth` y `min_samples_leaf`.

La celda de abajo enumera la línea base y los cuatro candidatos, cada uno con su configuración inicial:

In [ ]:
candidates = {
    "Baseline (most frequent class)": DummyClassifier(strategy="most_frequent"),
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(n_neighbors=k_neighbors),
    "Decision tree": DecisionTreeClassifier(
        max_depth=5, min_samples_leaf=min_leaf, random_state=RANDOM_STATE
    ),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

### 5.2 La regla de selección

La regla se fija antes de ver cualquier puntuación: **el modelo con el AUC medio de validación cruzada más alto pasa al
ajuste.**

### 5.3 Ejecución del torneo

> **En `scikit-learn`.** `cross_val_score` ajusta una copia nueva del pipeline en los pliegues de ajuste de cada ronda
> y devuelve las puntuaciones de validación, una por pliegue. `n_jobs=-1` ejecuta las rondas en paralelo en
> cada núcleo del procesador.

La celda de abajo valida de forma cruzada a cada candidato con el objeto de pliegues `cv` y la métrica `SCORING`, y
conserva las cinco puntuaciones de los pliegues de cada uno:

In [ ]:
cv_scores = {}
for name, estimator in candidates.items():
    cv_scores[name] = cross_val_score(
        make_model(estimator), X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
    )

La siguiente celda convierte las puntuaciones de los pliegues en una sola tabla, con la media y la desviación estándar de cada
modelo, ordenados de mejor a peor media:

In [ ]:
tournament = pd.DataFrame.from_dict(
    cv_scores,
    orient="index",
    columns=[f"auc_fold_{i}" for i in range(N_FOLDS)]
)
fold_columns = tournament.columns
tournament["cv_auc_mean"] = tournament[fold_columns].mean(axis=1)
tournament["cv_auc_std"] = tournament[fold_columns].std(axis=1)
tournament = tournament.sort_values("cv_auc_mean", ascending=False)
tournament

Todos los modelos están muy por encima del 0.5 de la línea base. Gradient boosting tiene la media más alta, 0.8477,
seguido por la regresión logística con 0.8461, KNN con 0.8369 y el árbol de decisión con 0.8303.

La siguiente celda aplica la regla de selección e imprime el modelo que pasa al ajuste:

In [ ]:
WINNER = tournament.index[0]
print(f"model chosen for tuning: {WINNER}")

Gradient boosting pasa a la Sección 6.

✍️ **Entregable 5 de 8 — El torneo**

- **Sus configuraciones iniciales**: para cada familia, la regla, el valor que otorgó en sus datos y la
  confirmación de que ningún valor se modificó después de ver una puntuación: *...*
- **Su regla de selección**, tal como la aplicó: *...*
- **El líder y su margen sobre el siguiente modelo**: *...*
- **El modelo que lleva al ajuste** y por qué: *...*
- **La línea base**: qué tan por encima de ella se encuentra cada modelo: *...*

💡 **Ejemplo** — Entregable 5 para los datos de cancelación. Elimínelo antes de entregar.

- **Configuraciones iniciales**: regresión logística y gradient boosting con sus valores predeterminados de biblioteca; KNN
  con $k = \sqrt{n}$ redondeado a impar, 67; el árbol con profundidad 5 y al menos 1% de $n$ por hoja, 45
  filas. Todas fijadas antes de que se ejecutara el torneo y ninguna modificada después.
- **Regla**: el AUC medio de validación cruzada más alto pasa al ajuste.
- **Líder**: gradient boosting, 0.8477, superando a la regresión logística (0.8461) por 0.0016.
- **Elegido**: gradient boosting, según la regla.
- **Línea base**: 0.5; el modelo real más débil, el árbol de decisión, está 0.33 por encima de ella.

## 6. Ajuste del Modelo Seleccionado

🗑️ **Guía — Sección 6.** Elimine antes de entregar.

Esta sección ajusta el modelo elegido en la Sección 5 mediante un estudio de **Optuna**, como en la sesión 6. Cada
prueba (trial) se evalúa con el mismo `make_model`, el mismo objeto de pliegues `cv` y el mismo `SCORING` del
torneo.

Los pasos:

1. Escribir la función objetivo: construye el modelo a partir de los valores que sugiere Optuna y devuelve la puntuación
   media de validación cruzada.
2. Ejecutar un estudio con un muestreador (sampler) que tenga semilla.
3. Comparar cada mejor valor con su rango. Un valor en un extremo (edge) significa que el óptimo puede estar fuera del
   rango; ampliarlo y ejecutar un nuevo estudio es la solución.
4. Validar de forma cruzada las mejores configuraciones y compararlas con el modelo sin ajustar.

**Rangos iniciales para cada familia**, para redactar su propio objetivo:

| Familia | Hiperparámetro | Rango | Llamada de Optuna |
|---|---|---|---|
| Regresión logística | `C` | 0.001 – 100 | `suggest_float(..., log=True)` |
| | `l1_ratio` | 0 – 1 | `suggest_float`; configurar `solver="saga"`, `max_iter=5000` |
| KNN | `n_neighbors` | 3 – 150 | `suggest_int` |
| | `weights` | `"uniform"`, `"distance"` | `suggest_categorical` |
| Árbol de decisión | `max_depth` | 2 – 20 | `suggest_int` |
| | `min_samples_leaf` | 1 – 100 | `suggest_int` |
| Gradient boosting | `learning_rate` | 0.01 – 0.5 | `suggest_float(..., log=True)` |
| | `n_estimators` | 30 – 400 | `suggest_int` |
| | `max_depth` | 2 – 5 | `suggest_int` |
| | `subsample` | 0.6 – 1.0 | `suggest_float` |
| | `min_samples_leaf` | 1 – 60 | `suggest_int` |
| Bosque aleatorio | `n_estimators` | 100 – 500 | `suggest_int` |
| | `max_features` | 0.1 – 1.0 | `suggest_float` |
| | `min_samples_leaf` | 1 – 50 | `suggest_int` |

🔧 **Para sus datos:** el estimador y las líneas `suggest_` en el objetivo, tomadas de la tabla. Conserve
la semilla, `cv`, `SCORING` y al menos 30 pruebas.

🎯 **Lo que verificamos:**
- El mismo objeto de pliegues y la misma métrica que en el torneo.
- Un muestreador con semilla.
- Cada rango indicado en el Entregable 6, con su justificación.
- Mejores valores ubicados en un extremo de su rango mencionados en el Entregable 6.
- Comparación entre ajustado y sin ajustar.
- El conjunto de prueba sigue sin utilizarse.

### 6.1 La función objetivo

> **En `optuna`.** `trial.suggest_float(name, low, high, log=True)` extrae un número decimal en una escala logarítmica;
> `trial.suggest_int(name, low, high)` extrae un número entero. Ambos incluyen `low` y `high`.

La siguiente celda define el objetivo. Optuna lo llama una vez por prueba: construye gradient boosting a partir de
los valores sugeridos y devuelve el AUC medio de validación cruzada en el conjunto de entrenamiento:

In [ ]:
def objective(trial: optuna.Trial) -> float:
    """AUC validado de forma cruzada de gradient boosting para un conjunto de hiperparámetros.

    Args:
        trial: La prueba de Optuna que propone los valores de los hiperparámetros.

    Returns:
        El ROC AUC medio a través de los pliegues de cv en el conjunto de entrenamiento.
    """
    estimator = GradientBoostingClassifier(
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        n_estimators=trial.suggest_int("n_estimators", 30, 400),
        max_depth=trial.suggest_int("max_depth", 2, 5),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 60),
        random_state=RANDOM_STATE,
    )
    scores = cross_val_score(
        make_model(estimator), X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
    )
    return scores.mean()

### 6.2 El estudio

La celda de abajo crea el estudio: la dirección a optimizar y el muestreador con su semilla fija para
que la ejecución se repita exactamente:

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

La siguiente celda ejecuta 30 pruebas, cada una con cinco ajustes. Optuna imprime una línea por prueba a medida que finaliza:

In [ ]:
N_TRIALS = 30

study.optimize(objective, n_trials=N_TRIALS)

La siguiente celda imprime la mejor puntuación, la prueba que la alcanzó y sus configuraciones:

In [ ]:
print(f"best CV AUC: {study.best_value:.4f} (trial {study.best_trial.number})")
pd.DataFrame([study.best_params])

El estudio alcanza 0.8506, por encima de los 0.8477 sin ajustar. La mejor prueba es la última,
la número 29.

> Observe cómo `max_depth` se encuentra en su límite inferior, 2, por lo que la profundidad óptima podría estar por debajo del rango. Una solución para esto es ejecutar un nuevo estudio que amplíe el rango de profundidad para comenzar en 1.

### 6.3 Modelo ajustado frente a sin ajustar

La celda de abajo construye gradient boosting con los mejores parámetros y lo valida de forma cruzada en los mismos
pliegues, junto a la fila sin ajustar del torneo:

In [ ]:
tuned_model = make_model(
    GradientBoostingClassifier(**study.best_params, random_state=RANDOM_STATE)
)
tuned_scores = cross_val_score(
    tuned_model, X_train, y_train, cv=cv, scoring=SCORING, n_jobs=-1
)

pd.DataFrame(
    {
        "cv_auc_mean": [tournament.loc[WINNER, "cv_auc_mean"], tuned_scores.mean()],
        "cv_auc_std": [tournament.loc[WINNER, "cv_auc_std"], tuned_scores.std(ddof=1)],
    },
    index=["untuned (Section 5)", "tuned (Section 6)"],
).round(4)

El ajuste eleva el AUC de validación cruzada de 0.8477 a 0.8506, una ganancia de 0.0029.

✍️ **Entregable 6 de 8 — El ajuste**

- **El objetivo**: cada hiperparámetro, su rango y escala, y por qué: *...*
- **Pruebas y semilla**: *...*
- **Verificación de extremos**: los valores que se ubicaron en un extremo y lo que hizo o haría al respecto: *...*
- **Ajustado frente a sin ajustar**: ambas puntuaciones y la ganancia: *...*

💡 **Ejemplo** — Entregable 6 para los datos de cancelación. Elimínelo antes de entregar.

- **Objetivo**: gradient boosting, con los cinco rangos de la tabla de la guía: `learning_rate`
  0.01–0.5 en escala logarítmica, porque los valores útiles abarcan más de una potencia de diez; `n_estimators`
  30–400; `max_depth` 2–5; `subsample` 0.6–1.0; `min_samples_leaf` 1–60.
- **Pruebas y semilla**: 30 pruebas, muestreador TPE con semilla `RANDOM_STATE` (42).
- **Verificación de extremos**: `max_depth` se ubica en su límite inferior, 2. El siguiente paso sería un nuevo estudio con el
  rango de profundidad comenzando en 1.
- **Ajustado frente a sin ajustar**: 0.8477 sin ajustar, 0.8506 ajustado, una ganancia de 0.0029.

## 7. La Prueba Final

🗑️ **Guía — Sección 7.** Elimine antes de entregar.

Utilice el conjunto de prueba una sola vez, aquí: reajuste el modelo ajustado en todo el conjunto de entrenamiento y evalúelo.

Compare la puntuación de prueba con la estimación validada de forma cruzada:

- **dentro de aproximadamente una desviación estándar (dispersión) de los pliegues**: la estimación se sostiene;
- **mucho más baja**: verifique si hay fuga de datos;
- **mucho más alta**: las filas de prueba pueden ser fáciles por casualidad.

No modifique el modelo después de ver la puntuación de prueba: el conjunto de prueba habría ayudado a elegir el
modelo, y su puntuación dejaría de ser una estimación honesta. Si lo hace, menciónelo en sus conclusiones.

🎯 **Lo que verificamos:**
- Una sola evaluación del modelo elegido en el conjunto de prueba, después de tomar todas las decisiones. Busque
  `X_test` en su cuaderno: aparece en la Sección 3 y aquí, en ninguna otra parte.
- La comparación con la estimación validada de forma cruzada.
- El Entregable 7 interpreta la brecha correctamente.

La celda de abajo reajusta el modelo optimizado en todo el conjunto de entrenamiento, lo evalúa una vez en el conjunto de prueba e
imprime el resultado junto a la estimación validada de forma cruzada:

In [ ]:
final_model = tuned_model.fit(X_train, y_train)
test_scores = final_model.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, test_scores)

print("ROC AUC of the final model")
print(f"CV mean estimate: {tuned_scores.mean():.4f}")
print(f"CV spread: {tuned_scores.std(ddof=1):.4f}")
print(f"test set: {test_auc:.4f}")

El modelo final obtiene una puntuación de **0.8455** en el conjunto de prueba, frente a una estimación validada de forma cruzada de 0.8506
con una dispersión entre pliegues de 0.0135.

### La curva ROC

Cada punto de la **curva ROC** representa un umbral. Grafica la proporción de clientes que se van que dicho
umbral detecta (la **tasa de verdaderos positivos**) frente a la proporción de clientes que se quedan que marca por
error (la **tasa de falsos positivos**). El AUC es el área bajo la curva; la diagonal representa la clasificación
por azar, con un área de 0.5.

La siguiente celda dibuja la curva ROC del modelo final en el conjunto de prueba, con la diagonal como referencia:

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.5))
RocCurveDisplay.from_predictions(y_test, test_scores, name="Final model", ax=ax)
ax.plot([0, 1], [0, 1], color="crimson", linestyle="--", linewidth=1, label="random sorting (AUC 0.5)")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("Final model on the test set")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()

La curva asciende pronunciadamente al principio: los clientes a los que el modelo otorga mayor puntuación son en su mayoría canceladores reales.
Luego se aplana, a medida que los canceladores restantes son más difíciles de separar de los clientes que se quedan. Dónde
detenerse a lo largo de la curva es una decisión de umbral, y pertenece al negocio, no a este cuaderno.

✍️ **Entregable 7 de 8 — La prueba final**

- **El modelo final**: su familia y sus hiperparámetros: *...*
- **Su puntuación de prueba**, junto a la estimación validada de forma cruzada y la dispersión de los pliegues: *...*
- **Qué significa la brecha**: *...*
- **Confirme que el conjunto de prueba se usó únicamente aquí** y que nada se modificó después de ver esta puntuación: *...*

💡 **Ejemplo** — Entregable 7 para los datos de cancelación. Elimínelo antes de entregar.

- **Modelo final**: gradient boosting con `learning_rate` 0.037, 207 árboles de profundidad 2, `subsample`
  0.91 y al menos 24 filas por hoja, reajustado en las 5,634 filas de entrenamiento con el preprocesamiento
  de la Sección 4.
- **Puntuación de prueba**: 0.8455, frente a una estimación validada de forma cruzada de 0.8506 con una dispersión entre pliegues de 0.0135.
- **La brecha**: 0.0051 más baja en el conjunto de prueba, dentro del rango de una desviación estándar de los pliegues.
- **Uso del conjunto de prueba**: `X_test` aparece solo en la división y en esta sección. Nada se modificó después
  de calcular esta puntuación.

## 8. Conclusiones

🗑️ **Guía — Sección 8.** Elimine antes de entregar.

Escriba esta sección para un lector que lea **únicamente esta sección**: un gerente o el equipo que utilizaría
el modelo. Utilice palabras sencillas y los números que ya obtuvo anteriormente. No introduzca nuevos resultados aquí.

🎯 **Lo que verificamos:**
- Una recomendación clara.
- Rendimiento esperado en datos nuevos, indicado como una cifra honesta.
- Al menos dos limitaciones reales, siendo una de ellas el torneo con modelos sin ajustar.
- Un siguiente paso concreto.

✍️ **Entregable 8 de 8 — Conclusiones**

- **Recomendación**: qué modelo, para qué decisión: *...*
- **Qué esperar en datos nuevos**, en palabras sencillas: *...*
- **Limitaciones**, al menos dos: *...*
- **Qué haría a continuación**: *...*

💡 **Ejemplo** — Entregable 8 para los datos de cancelación. Elimínelo antes de entregar.

- **Recomendación**: utilizar el modelo de gradient boosting ajustado para ordenar a los clientes según su riesgo de
  irse, y entregar al equipo de retención la lista de arriba hacia abajo.
- **Qué esperar**: en clientes que el modelo no ha visto, clasificará a un cliente que se va por encima de un
  cliente que se queda aproximadamente el 85% de las veces (AUC en prueba de 0.8455).
- **Limitaciones**:
  1. El torneo comparó modelos con configuraciones iniciales, no ajustadas, y solo gradient boosting
     fue ajustado. La regresión logística quedó 0.0016 por detrás sin ajustar y nunca se ajustó.
  2. Los datos son una instantánea de los clientes. Si los precios o los servicios cambian, el modelo debe verificarse
     nuevamente con datos nuevos.
  3. El modelo ordena a los clientes pero no dice a cuántos contactar. Dicho umbral requiere el costo de
     una oferta y el valor de un cliente retenido.
- **Siguiente paso**: solicitar al negocio esos dos costos y elegir el umbral a partir de ellos. Luego verificar
  qué columnas impulsan la clasificación antes de que el equipo de retención dependa de ella.

🗑️ **Antes de entregar** — elimine también esta celda.

1. Cada ✍️ celda está respondida sobre **sus** datos, y los campos de encabezado entre corchetes están completos.
2. Cada celda 💡 y 🗑️ está eliminada, incluida esta.
3. Cada celda de markdown antes y después de una celda de código describe **sus** datos y **sus** números. No queda texto
   sobre cancelación.
4. Cada número en su texto aparece en una salida encima de él.
5. `X_test` aparece solo en la Sección 3 y la Sección 7.
6. *Reiniciar Kernel y Ejecutar Todas las Celdas* finaliza sin errores, y usted entrega el archivo **con sus
   outputs**.

**Qué sigue.** Un modelo elegido de esta manera sigue siendo una caja negra: sabemos qué tan bien ordena, pero no
por qué. La sesión de interpretabilidad parte exactamente de un modelo así y pregunta qué columnas impulsan sus
predicciones.